# 01 Calibration: cable loopback

Analyzes a session captured with `uv run pedalrig calibrate` (cable FX SEND → INPUT R).
Nothing is written to disk; rerun freely. Levels: dBFS = sine peak re full scale;
dBu = send-referred via the manual spec (0 dBFS ≈ +14.5 dBu), not voltmeter-verified.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

if Path.cwd().name == "notebooks":  # sessions/ lives in the project root
    os.chdir("..")

from pedalrig import analysis as A
from pedalrig import plots as P
from pedalrig.session import Session, latest_calibration

In [ ]:
path = latest_calibration()          # or: path = Path("sessions/<dir>")
sess = Session(path)
cal = A.calibration_summary(sess)
print(path)
for k in ("latency_samples", "latency_ms", "latency_stable", "polarity", "G_db", "max_clean_dbfs",
          "noise_dbfs", "dc", "bw_lo_3db", "bw_hi_1db", "bw_hi_3db", "ripple_20_20k_db",
          "ripple_minmax_db", "repeat_dev_db", "repeat_latency_equal"):
    if k in cal:
        print(f"{k:22} {cal[k]}")
print()
for k, v in cal["checks"].items():
    print(f"check {k:10} {v}")

## C2 Routing: fundamental level on every USB input (expect signal on In 8 only; 1–4 are patch outputs)

In [ ]:
for ch, v in cal.get("routing_dbfs", {}).items():
    print(f"USB In {ch}: {v:7.1f} dBFS")

## C3 Noise floor

In [ ]:
if "noise_psd" in cal["curves"]:
    print(f"RMS 20 Hz-20 kHz: {cal['noise_dbfs']:.1f} dBFS (sine ref), DC {cal['dc']:.2e}")
    _ = P.plot_psd(*cal["curves"]["noise_psd"], hum=cal["hum"], title="C3 loopback noise")

## C4 Latency & polarity

In [ ]:
if "latency_runs" in cal:
    print("per-run latency (samples):", [round(r, 2) for r in cal["latency_runs"]])
    print(f"used: {cal['latency_samples']} samples = {cal['latency_ms']:.3f} ms, "
          f"stable={cal['latency_stable']}, polarity={cal['polarity']:+d}")

## C5 Levels / linearity (1 kHz)

In [ ]:
if "levels" in cal:
    print(f"{'in dBFS':>8} {'in dBu':>7} {'out dBFS':>9} {'gain dB':>8} {'THD+N':>7} {'HD2':>7} {'HD3':>7}  clip")
    for r in cal["levels"]:
        print(f"{r['in_dbfs']:8.0f} {r['in_dbu']:7.1f} {r['out_dbfs']:9.2f} {r['gain_db']:8.3f} "
              f"{r['thdn_db']:7.1f} {r['hd2_db']:7.1f} {r['hd3_db']:7.1f}  {r['clipped']}")
    _ = P.plot_levels(cal["levels"], cal["G_db"])

## C6 Wideband response (10 Hz–23 kHz sweep) and loop distortion

In [ ]:
if "wide_fr" in cal["curves"]:
    f, mag, ph = cal["curves"]["wide_fr"]
    _ = P.plot_fr([("loop", f, mag, ph)], title="C6 loop response re 1 kHz", lo=5, hi=f[-1], ylim=(-12, 3))
    _ = P.plot_fr([("loop", f, mag, None)], title="C6 zoom 20 Hz-20 kHz", ylim=(-1.5, 1.5))
    _ = P.plot_hd([("loop -20 dBFS", cal["curves"]["wide_hd"])], title="C6 loop harmonic distortion")

## C8 Repeatability

In [ ]:
if "repeat" in cal["curves"]:
    grid, deltas = cal["curves"]["repeat"]
    _ = P.plot_repeat(grid, {k: v for k, v in deltas.items() if k != "ref_sweep_m20"})

## What the numbers mean

- **Latency / polarity**: the fixed round trip of USB → FX SEND → INPUT R → USB. Pedal captures are aligned with it; a stable value (≤ 1 sample spread) means no per-capture alignment is needed.
- **G (loop gain)**: pedal captures are divided by G, so pedal output reads in the same units as the stimulus at the send jack ("send-referred").
- **max clean level**: highest stimulus whose loopback stays below −60 dB THD+N. Pedal distortion at or above this level is partly the interface's own.
- **Noise / hum**: floor under every measurement; distortion residuals near this level are not meaningful.
- **Bandwidth**: the Ampero DSP runs at 44.1 kHz internally, so harmonics above the −3 dB corner are not measurable; the pedal analysis cuts them off there.
- **Repeatability**: smallest frequency-response difference that is real rather than measurement noise.